In [ ]:
import re
from pathlib import Path

import os

import math

import torch
import pymupdf

from datasets import Dataset, load_dataset, DatasetDict

import gc

from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

In [ ]:
# Automatically set working directory to project root (one level up from notebooks/)
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Current Working Directory:", Path.cwd())

# Domain
DOMAIN = "metformin"

# Base experiment directory
exp_dir = Path(f"./experiments/{DOMAIN}")

# Stage 1 paths
stage1_dir = exp_dir / "stage1_domain"
stage1_adapter = stage1_dir / "adapter"
stage1_merged = stage1_dir / "merged_model"

# # Stage 2 paths
# stage2_dir = exp_dir / "stage2_instruct"
# stage2_adapter = stage2_dir / "adapter"
# stage2_merged = stage2_dir / "merged_model"

# Create directories if they don't exist
stage1_dir.mkdir(parents=True, exist_ok=True)
# stage2_dir.mkdir(parents=True, exist_ok=True)

# Model and data
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
pdf_path = Path("data/Metformin.pdf")

# Training configuration
max_length = 256

# Hardware check
assert torch.cuda.is_available(), "CUDA GPU is required for this setup."

print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("PyTorch:", torch.__version__)

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract non-empty page text from a PDF."""
    pdf_path = Path(pdf_path)

    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    page_texts = []

    with pymupdf.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()

            if text:
                page_texts.append(text)

    return page_texts

def split_paragraphs(pages, min_length=30):
    """Split extracted page text on blank lines."""
    paragraphs = []

    for page_text in pages:
        chunks = re.split(r"\n\s*\n+", page_text)

        for chunk in chunks:
            clean = re.sub(r"\s+", " ", chunk).strip()

            if len(clean) >= min_length:
                paragraphs.append(clean)

    return paragraphs

In [ ]:
# Extract and create dataset
pdf_texts = extract_text_from_pdf(pdf_path)
paragraphs = split_paragraphs(pdf_texts)

if len(paragraphs) < 2:
    raise ValueError(f"Need at least 2 paragraphs to create a split, found {len(paragraphs)}.")

raw_dataset = Dataset.from_dict({"text": paragraphs})

# 75% train paragraphs, 25% validation paragraphs (here 3, 1 split)
split_dataset = raw_dataset.train_test_split(test_size=0.25, seed=42)
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"]
})

print(f"Total paragraphs: {len(paragraphs)}")
print(f"Train size: {len(dataset['train'])} | Validation size: {len(dataset['validation'])}")

In [ ]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
    )

tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
)

tokenized_train = tokenized["train"]
tokenized_val = tokenized["validation"]

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(f"Train samples: {len(tokenized_train)}")
print(f"Validation samples: {len(tokenized_val)}")

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.eval()

def evaluate_loss(model, eval_dataset):
    model.eval()
    loader = DataLoader(
        eval_dataset,
        batch_size=2,
        shuffle=False,
        collate_fn=data_collator,
    )

    total_loss = 0.0
    total_tokens = 0

    with torch.inference_mode():
        for batch in loader:
            batch = {k: v.to(model.device) for k, v in batch.items()}
            outputs = model(**batch)

            # Mask out padding tokens (-100) when counting valid target tokens
            valid_tokens = (batch["labels"][:, 1:] != -100).sum().item()
            total_loss += outputs.loss.item() * valid_tokens
            total_tokens += valid_tokens

    if total_tokens == 0:
        raise ValueError("Validation dataset contains no valid target tokens.")

    average_loss = total_loss / total_tokens
    perplexity = math.exp(min(average_loss, 100))  # guard against overflow

    return {
        "loss": average_loss,
        "perplexity": perplexity,
        "tokens": total_tokens,
    }

baseline_results = evaluate_loss(base_model, tokenized_val)

print(f"Baseline validation loss: {baseline_results['loss']:.4f}")
print(f"Baseline perplexity:      {baseline_results['perplexity']:.2f}")
print(f"Evaluated tokens:         {baseline_results['tokens']}")

In [ ]:
# 1. Prepare quantized base model for LoRA
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

q_lora_model = get_peft_model(base_model, lora_config)
q_lora_model.print_trainable_parameters()

# 2. Training arguments tuned for 3 paragraphs
training_args = TrainingArguments(
    output_dir=str(stage1_dir / "checkpoints"),
    num_train_epochs=15,                   # ~45 steps total so the model absorbs the jargon
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=3e-4,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=1,                       # Log every step so you see progress
    save_total_limit=1,
    report_to="none",
    seed=42,
)

# 3. Trainer with both splits provided
trainer = Trainer(
    model=q_lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

train_result = trainer.train()

# Save adapter and tokenizer
trainer.save_model(str(stage1_adapter))
tokenizer.save_pretrained(str(stage1_adapter))
q_lora_model.config.use_cache = True

print(f"\nAdapter saved to: {stage1_adapter}")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")

In [ ]:
post_results = evaluate_loss(q_lora_model, tokenized_val)

print("=" * 45)
print(f"{'Metric':<15}{'Baseline':<15}{'Fine-tuned':<15}")
print("-" * 45)
print(f"{'Loss:':<15}{baseline_results['loss']:<15.4f}{post_results['loss']:<15.4f}")
print(f"{'Perplexity:':<15}{baseline_results['perplexity']:<15.2f}{post_results['perplexity']:<15.2f}")
print("=" * 45)

In [ ]:
# 1. Clean up training objects from VRAM
del base_model, q_lora_model, trainer
gc.collect()
torch.cuda.empty_cache()

# 2. Reload base model in FP16 with minimal CPU RAM overhead
print("Loading base model in FP16...")
clean_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cuda",
)

# 3. Attach trained LoRA adapter and merge into base weights
print("Merging LoRA weights into base model...")
lora_model = PeftModel.from_pretrained(clean_base, str(stage1_adapter))
merged_model = lora_model.merge_and_unload()

# 4. Save merged standalone model
print(f"Saving merged model to {stage1_merged}...")
merged_model.save_pretrained(stage1_merged, safe_serialization=True)
tokenizer.save_pretrained(stage1_merged)

# 5. Sanity check: test generation on the merged model
prompt = (
    "Metformin is commonly used to treat type 2 diabetes. "
    "Its mechanism of action involves"
)

inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)

with torch.inference_mode():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.pad_token_id,
    )

generated_tokens = outputs[0, inputs["input_ids"].shape[-1]:]
print("\n--- Prompt ---")
print(prompt)
print("\n--- Merged Model Output ---")
print(tokenizer.decode(generated_tokens, skip_special_tokens=True))

In [ ]:
merged_path = "./experiments/metformin/stage1_domain/merged_model"

print(f"Loading merged medical model from {merged_path}...")
tokenizer = AutoTokenizer.from_pretrained(merged_path)
model = AutoModelForCausalLM.from_pretrained(
    merged_path,
    torch_dtype=torch.float16,
    device_map="cuda",
)
model.eval()

prompt = "Metformin is commonly used to treat type 2 diabetes. Its mechanism of action involves"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.pad_token_id,
    )

continuation = tokenizer.decode(outputs[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("\n--- Prompt ---")
print(prompt)
print("\n--- Continuation ---")
print(continuation)